# Personalized Learning Assistant

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashyx12/agentic-ai-assignment/blob/main/personalized_learning_assistant.ipynb)

**Project:** Agentic AI-Based Personalized Learning and Student Assistance System.

This small demo uses three specialist components—**Assessor → Learning Planner → Study Coach**—to tailor a revision plan to a learner's quiz answers. The core runs in Colab without any API key.

## 1. Quiz and learning resources
Edit `learner_answers` in the next section to simulate a student's answers. The example uses introductory Python topics.

In [ ]:
import pandas as pd

QUESTIONS = [
    {"topic": "Python basics", "question": "What does len([2, 4, 6]) return?", "options": ["2", "3", "6"], "answer": "3"},
    {"topic": "Loops", "question": "How many times does range(3) iterate?", "options": ["2", "3", "4"], "answer": "3"},
    {"topic": "Functions", "question": "Which keyword defines a function?", "options": ["func", "def", "function"], "answer": "def"},
    {"topic": "Lists", "question": "Which list method adds an item to the end?", "options": ["append()", "push()", "insert_end()"], "answer": "append()"},
    {"topic": "Dictionaries", "question": "A dictionary stores data as…", "options": ["key-value pairs", "only numbers", "equations"], "answer": "key-value pairs"},
]

LEARNING_RESOURCES = {
    "Python basics": ("https://docs.python.org/3/tutorial/", "Review types, expressions, and built-in functions."),
    "Loops": ("https://docs.python.org/3/tutorial/controlflow.html", "Write a loop that processes a list."),
    "Functions": ("https://docs.python.org/3/tutorial/controlflow.html#defining-functions", "Write a function with parameters and a return value."),
    "Lists": ("https://docs.python.org/3/tutorial/datastructures.html", "Practice filtering and sorting a list."),
    "Dictionaries": ("https://docs.python.org/3/tutorial/datastructures.html#dictionaries", "Count word frequencies using a dictionary."),
}

display(pd.DataFrame([{"Topic": q["topic"], "Question": q["question"], "Options": " | ".join(q["options"])} for q in QUESTIONS]))

## 2. Enter learner answers
Use the exact option text. Change values to test how the generated plan changes.

In [ ]:
learner_answers = {
    "Python basics": "3",
    "Loops": "2",                 # intentionally incorrect example
    "Functions": "def",
    "Lists": "append()",
    "Dictionaries": "only numbers", # intentionally incorrect example
}

## 3. Run the agent workflow
Each component has one responsibility and passes structured results to the next component.

In [ ]:
def assessor(questions, answers):
    """Agent 1: score answers and identify topics to revise."""
    scores = {q['topic']: answers.get(q['topic']) == q['answer'] for q in questions}
    weak_topics = [topic for topic, correct in scores.items() if not correct]
    return {"scores": scores, "weak_topics": weak_topics}

def learning_path_planner(assessment, resources):
    """Agent 2: prioritize weak topics and attach resources and practice tasks."""
    topics = assessment['weak_topics'] or list(assessment['scores'].keys())
    plan = []
    for day, topic in enumerate(topics, start=1):
        url, task = resources[topic]
        plan.append({"Day": day, "Topic": topic, "Learning resource": url, "Practice task": task})
    plan.append({"Day": len(plan) + 1, "Topic": "Review", "Learning resource": "Your class notes", "Practice task": "Retake the quiz and explain each answer in your own words."})
    return plan

def study_coach(assessment, plan):
    """Agent 3: summarize the result and recommend the next step."""
    score = sum(assessment['scores'].values())
    total = len(assessment['scores'])
    focus = ', '.join(assessment['weak_topics']) if assessment['weak_topics'] else 'No weak topics found in this short quiz'
    return f"Quiz score: {score}/{total}. Revision focus: {focus}. Follow the plan in order, then retake the quiz."

assessment = assessor(QUESTIONS, learner_answers)
study_plan = learning_path_planner(assessment, LEARNING_RESOURCES)
print(study_coach(assessment, study_plan))

print("\nAssessment results:")
display(pd.DataFrame([{"Topic": topic, "Result": "Correct" if correct else "Revise"} for topic, correct in assessment['scores'].items()]))

print("\nPersonalized learning plan:")
display(pd.DataFrame(study_plan))

## 4. Summary
- The assessor finds topics with incorrect answers.
- The planner creates a prioritized revision schedule with resources and practice tasks.
- The study coach summarizes the assessment and gives a next step.

**Cost:** the core workflow needs only Python and pandas and does not call a hosted LLM API. This is a small rule-based, multi-component prototype—not a production system or autonomous LLM-agent framework.

## 4. Optional: generate study advice with a local model
Run this section only if you want a generated explanation. It downloads the small public `google/flan-t5-small` model into the Colab runtime; **no OpenAI/Gemini API key is needed**. You can skip it and still use the personalized plan above.

In [ ]:
!pip -q install transformers sentencepiece
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
study_coach_model = pipeline("text2text-generation", model="google/flan-t5-small", device=device)
topics = ", ".join(assessment["weak_topics"] or list(assessment["scores"].keys()))
prompt = f"You are a friendly study coach. Create a short 3-step study plan for a student revising: {topics}. Be specific and concise."
generated = study_coach_model(prompt, max_new_tokens=100, do_sample=False)[0]["generated_text"]
print(generated)

## 5. Summary
- The assessor finds topics with incorrect answers.
- The planner creates a prioritized revision schedule with resources and practice tasks.
- The study coach summarizes the assessment and gives a next step.
- An optional small Hugging Face model can generate additional advice directly inside Colab.

**Cost:** the core workflow uses Python and pandas only; the optional model downloads public model weights and runs in Colab. Neither requires a paid API key. This is a small educational prototype, not a production tutor.